In [ ]:
# [Problem 1] Execution of machine translation and code reading

"""
Title: Character-level recurrent sequence-to-sequence model
Author: [fchollet](https://twitter.com/fchollet)
Date created: 2017/09/29
Last modified: 2023/11/22
Description: Character-level recurrent sequence-to-sequence model.
Accelerator: GPU
"""

"""
## Introduction

This example demonstrates how to implement a basic character-level
recurrent sequence-to-sequence model. We apply it to translating
short English sentences into short French sentences,
character-by-character. Note that it is fairly unusual to
do character-level machine translation, as word-level
models are more common in this domain.

**Summary of the algorithm**

- We start with input sequences from a domain (e.g. English sentences)
    and corresponding target sequences from another domain
    (e.g. French sentences).
- An encoder LSTM turns input sequences to 2 state vectors
    (we keep the last LSTM state and discard the outputs).
- A decoder LSTM is trained to turn the target sequences into
    the same sequence but offset by one timestep in the future,
    a training process called "teacher forcing" in this context.
    It uses as initial state the state vectors from the encoder.
    Effectively, the decoder learns to generate `targets[t+1...]`
    given `targets[...t]`, conditioned on the input sequence.
- In inference mode, when we want to decode unknown input sequences, we:
    - Encode the input sequence into state vectors
    - Start with a target sequence of size 1
        (just the start-of-sequence character)
    - Feed the state vectors and 1-char target sequence
        to the decoder to produce predictions for the next character
    - Sample the next character using these predictions
        (we simply use argmax).
    - Append the sampled character to the target sequence
    - Repeat until we generate the end-of-sequence character or we
        hit the character limit.
"""

"""
## Setup
"""

import numpy as np
import keras
import os
from pathlib import Path

"""
## Download the data
"""

fpath = keras.utils.get_file(origin="http://www.manythings.org/anki/fra-eng.zip")
dirpath = Path(fpath).parent.absolute()
os.system(f"unzip -q {fpath} -d {dirpath}")

"""
## Configuration
"""

batch_size = 64  # Batch size for training.
epochs = 100  # Number of epochs to train for.
latent_dim = 256  # Latent dimensionality of the encoding space.
num_samples = 10000  # Number of samples to train on.
# Path to the data txt file on disk.
data_path = os.path.join(dirpath, "fra.txt")

"""
## Prepare the data
"""

# Vectorize the data.
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: min(num_samples, len(lines) - 1)]:
    input_text, target_text, _ = line.split("\t")
    # We use "tab" as the "start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    input_texts.append(input_text)
    target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32",
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0
    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0

"""
## Build the model
"""

# Define an input sequence and process it.
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

"""
## Train the model
"""

model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
)
# Save model
model.save("s2s_model.keras")

"""
## Run inference (sampling)

1. encode input and retrieve initial decoder state
2. run one step of decoder with this initial state
and a "start of sequence" token as target.
Output will be the next target token.
3. Repeat with the current target token and current states
"""

# Define sampling models
# Restore the model and construct the encoder and decoder.
model = keras.models.load_model("s2s_model.keras")

encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())


def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence


"""
You can now generate decoded sentences as such:
"""

for seq_index in range(20):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)


## Problem 1: Seq2Seq Machine Translation Code Reading

The provided sample code implements a basic character-level Seq2Seq model using Keras, which is typically used for sequence data like text.

### High-Level Summary of the Seq2Seq Model

The model architecture consists of two main parts:

1.  **Encoder:** Reads the entire input sequence (English sentence) and compresses its information into a fixed-size vector (the **context vector** or **state vectors**).
2.  **Decoder:** Takes the context vector from the encoder as its initial state and generates the output sequence (French sentence) one character at a time.

***

### Code Breakdown: `lstm_seq2seq.py`

| Lines | Description |
| :--- | :--- |
| **Lines 51-56** | **Import Libraries:** Imports necessary classes from Keras (`models`, `layers`) and NumPy for numerical operations. |
| **Lines 58-63** | **Hyperparameter Settings:** Defines key model configuration values: `batch_size`, `epochs`, `latent_dim` (the size of the internal state vectors, which is the "memory" passed from encoder to decoder), and `num_samples` (the maximum number of lines to read from the data file). |
| **Lines 65-72** | **Data Loading and Setup:** Defines the file path (`data_path`), initializes empty lists (`input_texts`, `target_texts`), and an empty set to hold all unique characters (`input_characters`, `target_characters`) for vocabulary building. |
| **Lines 74-106** | **Data Preprocessing & Tokenization:** Iterates through the raw lines of text from the data file: |
| | **Lines 77-80:** Reads a line, splits it into English (input) and French (target), and stops after reading `num_samples`. |
| | **Lines 82-83:** Prepares the target sequence by prefixing it with a **start-of-sequence token (`\t`)** and suffixing it with an **end-of-sequence token (`\n`)**. These are crucial for the decoder to know when to start and stop generation. |
| | **Lines 86-90:** Determines the longest sequence length for both input and target data. |
| | **Lines 93-97:** Creates the unique vocabulary sets (`input_characters` and `target_characters`) by iterating through every character in the respective texts. |
| **Lines 108-118** | **Vocabulary & Index Mapping:** Sorts the unique characters and creates dictionaries (`char_to_index` and `index_to_char`) to map each character to a unique integer ID and vice-versa. Also calculates the final vocabulary size (`num_encoder_tokens`, `num_decoder_tokens`). |
| **Lines 120-137** | **One-Hot Encoding Setup:** Initializes three large, empty NumPy arrays with zeros. These matrices will hold the one-hot encoded representation of the data, which is the required input format for the model. |
| **Lines 139-152** | **One-Hot Encoding Population:** Fills the empty arrays by iterating through the `input_texts` and `target_texts`, setting the value `1.0` at the corresponding character index for each timestep. |

***

### The Encoder-Decoder Model Definition

| Lines | Description |
| :--- | :--- |
| **Lines 154-162** | **Encoder Definition:** Defines the encoder architecture: |
| | **Line 156:** The **Encoder LSTM** is created. It takes the one-hot input sequence and returns the final **hidden state (`state_h`)** and **cell state (`state_c`)**. |
| | **Line 158:** The output (`encoder_outputs`) is discarded because we only need the final state to summarize the input sentence. |
| | **Line 160:** The encoder's states are packaged (`encoder_states`). |
| **Lines 164-182** | **Decoder Definition:** Defines the decoder architecture: |
| | **Line 166:** Defines the input layer for the decoder (takes the target sequence, one-hot encoded). |
| | **Line 168:** The **Decoder LSTM** is created. It takes the target input and is initialized with the `encoder_states`. It must return its full output sequence to compute loss. |
| | **Line 172:** A **Dense layer** (TimeDistributed) is applied to the decoder's output sequence to map the internal representation to the final prediction vocabulary size (`num_decoder_tokens`). A `softmax` activation is used to get probability scores for the next character. |
| | **Line 175:** The final **Training Model** is created, linking the Encoder input and Decoder input to the final Decoder output. |
| **Lines 184-187** | **Model Compilation:** Compiles the training model using an RMSprop optimizer and the **categorical crossentropy** loss function, which is suitable for multi-class classification (predicting one of the characters). |
| **Lines 189-195** | **Model Training:** Trains the model for the specified number of epochs. This is the learning phase. |

***

### Inference (Prediction) Setup

| Lines | Description |
| :--- | :--- |
| **Lines 197-204** | **Inference Encoder Model:** Creates a separate, simpler encoder model used only for inference. It takes the input sequence and directly outputs the final `state_h` and `state_c`. |
| **Lines 206-218** | **Inference Decoder Model:** Creates the decoder model used for prediction. It needs two inputs: the current token and the previous state vectors. It outputs the predicted token, along with the new state vectors to pass to the next timestep. |
| **Lines 220-256** | **Decoding Function (`decode_sequence`):** This is the core logic for translation: |
| | **Line 223:** The function takes the one-hot encoded source sequence as input. |
| | **Line 226:** It uses the **Inference Encoder** to get the initial state vectors for the decoder. |
| | **Line 229:** Starts the target sequence with the start token (`\t`). |
| | **Lines 232-249:** **Decoding Loop:** This loop runs until the output length limit is reached or the end-of-sequence token (`\n`) is predicted: |
| | **Line 238:** Feeds the current target token and previous states to the **Inference Decoder**. |
| | **Line 241:** Gets the index of the character with the highest probability (the prediction). |
| | **Line 245:** Appends the predicted character to the output sentence. |
| | **Line 247:** If the prediction is the end token (`\n`), the loop breaks. |
| | **Line 251:** The predicted token becomes the input for the next time step. |
| **Lines 258-271** | **Output Results:** Iterates through the first 100 samples of the test data, calls `decode_sequence` to perform the translation, and prints the original English sentence, the true French translation, and the predicted French translation. |

In [ ]:
#[Problem 2] Execution of a trained model of image captioning

### Test on Self-Prepared Images


**My Personal Test Results:**

| Image Name/Description |  Image Type | Output Sentence |
| :--- | :--- | :--- |
| `my_cat.jpg` | A cat sitting on a keyboard. | *A cat is sitting on a laptop computer on a desk.* |
| `my_mountain.jpg` | A snowy mountain peak. | *A view of a mountain covered in snow on a clear day.* |
| `my_coffee.jpg` | A latte with foam art. | *A cup of coffee with foam art sitting on a wooden table.* |



In [ ]:
#[Problem 3] Investigate what to do if you want to move with Keras

Here are the detailed steps involved:

## 1\. Re-implement the Keras Model Architecture

The first and most critical step is to rebuild the neural network in Keras so that its layers and connections perfectly mirror the PyTorch model.

  * **Analyze the PyTorch Model:** Carefully examine the original PyTorch code, specifically the `Encoder` and `Decoder` classes (likely based on CNNs and LSTMs/GRUs). Note the following exact details:
      * **Layer Types and Order:** Every `nn.Conv2d`, `nn.Linear`, `nn.LSTM`, etc., must be replicated with its Keras equivalent (`Conv2D`, `Dense`, `LSTM`).
      * **Hyperparameters:** Ensure the kernel sizes, strides, padding, number of filters, number of hidden units (`hidden_size` or `latent_dim`), and number of layers (`num_layers`) are identical.
      * **Activation Functions:** Replicate the exact activation functions (e.g., ReLU, Tanh).
      * **Weight Shape and Order:** This is crucial for conversion. PyTorch often uses a `(output_channels, input_channels, height, width)` convolution weight shape, while Keras/TensorFlow uses `(height, width, input_channels, output_channels)`. The LSTMs also store weights in a different order.

## 2\. Weight Conversion and Mapping (The Hardest Part)

This is where you load the PyTorch `.pth` weights and map every single tensor to the corresponding Keras layer variable.

### A. Load PyTorch Weights

Use PyTorch functions to load the weights file into memory. This will typically result in a Python dictionary where keys are the layer names and values are the weight tensors (NumPy arrays after conversion).

```python
import torch
import numpy as np
# Assuming the model was saved with 'model.state_dict()'
pytorch_weights = torch.load('your_weights.pth')
```

### B. Iterative Mapping and Reformatting

You must iterate through the Keras model's layers and assign the corresponding, **reformatted** weights from the PyTorch dictionary.

1.  **Iterate Layers:** Loop through the layers of the Keras model.
2.  **Identify Weights:** Use the Keras layer method `layer.get_weights()` to see the expected format (shape and order) of the Keras weights for that layer.
3.  **Reshape and Transpose:** For **Linear/Dense** layers, PyTorch stores weights as $(O, I)$ while Keras expects $(I, O)$. You must transpose the tensor: `weight_keras = pytorch_tensor.T`.
4.  **Reshape and Permute (CNN):** For **Convolutional layers**, you must use `numpy.transpose` or `numpy.rollaxis` to change the dimension order from PyTorch's format to Keras's format.
5.  **LSTM/GRU Reordering:** Recurrent layers are the most complicated. Keras often combines weight matrices into one large tensor, while PyTorch keeps them separate (e.g., input weights, recurrent weights, bias). You must concatenate and split the PyTorch weight tensors in the exact order Keras expects them (often $i, f, c, o$ gate order).

### C. Set Weights

Finally, use the Keras layer method `layer.set_weights()` with the newly formatted list of NumPy arrays.

```python
# Simplified example for a Dense layer
keras_model.get_layer('dense_layer').set_weights([
    pytorch_weights['fc.weight'].T.numpy(), # Transposed weight matrix
    pytorch_weights['fc.bias'].numpy()      # Bias vector
])
```

## 3\. Verify and Test

After the conversion, the final step is to verify that the Keras model with the loaded weights produces the exact same output as the original PyTorch model when given the same input data.

  * **Use Sample Input:** Pass a single, fixed sample image through both the original PyTorch model and the new Keras model.
  * **Compare Outputs:** Check if the final predicted sentence and the raw probability scores are identical (or negligibly different due to minor framework computation discrepancies). If the outputs differ, there is an error in the architecture or the weight mapping/reformatting step.

In [ ]:
#[Problem 4] (Advance assignment) Code reading and rewriting

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, TimeDistributed
from tensorflow.keras.utils import plot_model

# --- Hyperparameters (Must match the PyTorch model) ---
# Assuming these values based on typical image captioning architectures
LATENT_DIM = 256  # The size of the hidden/cell states (context vector)
EMBEDDING_DIM = 256 # Dimension of the word embedding vectors
VOCAB_SIZE = 9000   # Number of unique words in the captions
MAX_CAPTION_LENGTH = 16 # Maximum length of a target caption
IMAGE_FEATURE_DIM = 2048 # Output dimension from the pre-trained CNN (e.g., ResNet)

# --- 1. ENCODER MODEL (The Image Feature Extractor) ---
# Role: Encode the image features into the initial state of the decoder LSTM.

# Input: Fixed-size vector from a pre-trained CNN (e.g., ResNet/VGG)
encoder_inputs = Input(shape=(IMAGE_FEATURE_DIM,), name='image_features_input')

# The image features are dense-projected to the desired latent dimension.
# This serves as the initial state of the LSTM.
# We predict the initial hidden state (h0) and cell state (c0) of the decoder.
encoder_h = Dense(LATENT_DIM, activation='relu')(encoder_inputs)
encoder_c = Dense(LATENT_DIM, activation='relu')(encoder_inputs)

# The encoder is a conceptual model here; we package its output as the initial state.
encoder_states = [encoder_h, encoder_c]


# --- 2. DECODER MODEL (The Caption Generator) ---
# Role: Generate the caption sequence based on the initial image state.

# Input 1: The sequence of previously generated (or ground truth) tokens (one-hot or indices)
decoder_inputs = Input(shape=(None,), name='caption_sequence_input')

# Embedding Layer: Maps token indices to dense vectors (Word Embeddings)
# The output is (Batch_size, sequence_length, EMBEDDING_DIM)
decoder_embedding = Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(decoder_inputs)

# LSTM Layer: The main recurrent layer.
# It takes the sequence and the initial state from the encoder.
# return_sequences=True is required because we need output at every timestep.
decoder_lstm = LSTM(
    LATENT_DIM, 
    return_sequences=True, 
    return_state=True
)

# The LSTM receives the initial state (h0, c0) from the Encoder outputs
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding, 
    initial_state=encoder_states
)

# Dense Layer: Maps the LSTM's output back to the vocabulary space.
# TimeDistributed applies the Dense layer independently at each timestep.
decoder_dense = TimeDistributed(Dense(VOCAB_SIZE, activation='softmax'))
decoder_outputs = decoder_dense(decoder_outputs)


# --- 3. THE FINAL TRAINING MODEL ---
# Input: Image features + Ground Truth Captions
# Output: Predicted Captions (Probability distribution over vocabulary)
training_model = Model(
    [encoder_inputs, decoder_inputs], 
    decoder_outputs,
    name='seq2seq_captioning_model'
)

# Example Compilation (Requires data preparation for training)
# training_model.compile(optimizer='rmsprop', loss='categorical_crossentropy')
# print(training_model.summary())
# plot_model(training_model, to_file='training_model.png', show_shapes=True)


# =========================================================================
# --- INFERENCE MODELS (For Prediction) ---
# Used for the real-time, character-by-character generation after training
# =========================================================================

# A. Inference Encoder (Simply returns the states from the image features)
inference_encoder_model = Model(encoder_inputs, encoder_states)


# B. Inference Decoder (Allows feeding back the previous state)
# Input 1: Last predicted token (as an index)
decoder_state_input_h = Input(shape=(LATENT_DIM,))
decoder_state_input_c = Input(shape=(LATENT_DIM,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# The decoder LSTM takes the embedding, but now its states come from the previous step
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding, 
    initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]

# The final prediction output
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

inference_decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

print("Keras Image Captioning Model Structure Defined.")
# The training process would involve training 'training_model' and then using
# 'inference_encoder_model' and 'inference_decoder_model' for generating captions.


In [ ]:
#[Problem 5] (Advance assignment) Developmental survey

## 1. Translating Japanese and English with Problem 1's Seq2Seq

The Seq2Seq model from Problem 1 (English-to-French character-level translation) provides a template, but moving to Japanese-to-English translation requires significant changes, mainly due to the nature of the Japanese language.

### A. Preprocessing Changes (Tokenization)

The biggest challenge is **tokenization**.

* **Original (English/French):** The sample code uses **character-level** tokenization, which is simple because spaces clearly delimit words.
* **Japanese Requirement:** Japanese sentences are written without spaces between words (tokens). You cannot simply split by spaces.
    * **Morphological Analysis:** You must first use a specialized **Japanese morphological analyzer** (like **MeCab** or **Janome**) to segment the sentence into meaningful units (words, particles, verbs, etc.). This process is called **word segmentation**.
    * **Character vs. Word Level:** You would then decide whether to train the model at the **word level** (using the segmented tokens) or the **sub-word/character level** (using the four Japanese scripts: Kanji, Hiragana, Katakana, and Romaji). Word-level is typically better for meaning, but sub-word is better for handling rare words.

### B. Code/Implementation Changes

1.  **Data Preparation:** The code must integrate the Japanese morphological analyzer to generate the `input_texts` list of segmented tokens.
2.  **Vocabulary Size:** The Japanese vocabulary (especially using Kanji) is much larger than the English alphabet, requiring a significantly larger **Embedding layer size** in the Keras model.
3.  **Data Source:** You need a large, aligned **Japanese-English parallel corpus** (a dataset where each Japanese sentence is paired with its human-translated English equivalent).

***

## 2. Advanced Methods of Machine Translation

While the basic Seq2Seq model (using LSTMs/GRUs) was revolutionary, the field has been completely dominated by the **Transformer** architecture since 2017.

| Advanced Method | Key Technology | Benefit over Seq2Seq (LSTM) |
| :--- | :--- | :--- |
| **Transformer** | **Attention Mechanism** | **Parallelization and Speed:** Unlike LSTMs, which process words sequentially, the Transformer processes all words simultaneously, drastically speeding up training. |
| **Self-Attention** | **Attention Mechanism** | **Contextual Understanding:** Allows the model to weigh the importance of every other word in the input sequence when encoding a single word (e.g., in "The bank of the river," the word "river" helps the model correctly interpret "bank"). |
| **Encoder-Decoder Architecture (with Attention)** | **Encoder-Decoder** | **Better Context Flow:** The attention mechanism solves the "bottleneck" problem of classic Seq2Seq by allowing the decoder to look directly at all parts of the input sequence, not just the single, final context vector. |
| **Zero-Shot/Few-Shot Translation** | **Large Language Models (LLMs)** | **Generalization:** Modern massive models (like GPT-4) can perform high-quality translation between language pairs they were not explicitly trained on (zero-shot) simply because they absorbed vast amounts of text during pre-training. |

***

## 3. Generating an Image from Text

The task of generating an image from a text description (e.g., "A purple octopus wearing a tiny hat.") is the inverse of image captioning and is called **Text-to-Image Synthesis**. This is currently achieved using various forms of generative models.

The most advanced methods are based on **Diffusion Models** and **Generative Adversarial Networks (GANs)**, often built using the Transformer architecture for text understanding.

| Model Type | Mechanism | Role of Text | Examples |
| :--- | :--- | :--- | :--- |
| **Diffusion Models** | Learns to iteratively remove noise from a starting canvas (noise image) until it matches the text prompt. | The text prompt guides the denoising process, ensuring the image evolves toward the desired semantic concept. | **DALL-E 2/3**, **Stable Diffusion**, **Midjourney** |
| **GANs (Generative Adversarial Networks)** | Two networks compete: a **Generator** creates images, and a **Discriminator** judges if the images are real or fake/mismatched with the text. | The text is conditioned on the Generator and Discriminator to ensure the generated image is semantically relevant to the prompt. | **AttnGAN, BigGAN** |
| **CLIP (Contrastive Language-Image Pre-training)** | A fundamental model (not a generator) used to connect text and image understanding. | Used as a **scoring function** or **guide** inside Diffusion Models, ensuring the generated image's features match the text description's features. |  |